
Checkpoint: `ash56/ssl-aasist` (XLS-R 300M + AASIST), best of the public ones in


In [ ]:
# %pip install torch torchaudio soundfile pandas scikit-learn matplotlib huggingface_hub
import shutil, subprocess, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO = Path(".").resolve()
SONGS = REPO / "songs"
OUT = REPO / "songs_out"
LABELS = REPO / "songs_labels.csv"   # optional: file,label (1 = ai)

BATCH = 8
MAX_CHUNKS = 45      # 45 * 4 = 180 = 3 min 
MIN_RMS = 1e-3       # skip silent windows
RUN_TTA = True 

OUT.mkdir(exist_ok=True)
FFMPEG = shutil.which("ffmpeg")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(0)
np.random.seed(0)
print(DEVICE, FFMPEG)

## Checkpoint

In [ ]:
from huggingface_hub import hf_hub_download

ckpt_dir = REPO / "public_ckpt" / "ssl_aasist"
ckpt_dir.mkdir(parents=True, exist_ok=True)

for fn in ["pytorch_model.bin", "model_hf.py", "config.json", "config_ssl.py"]:
    if not (ckpt_dir / fn).exists():
        shutil.copy(hf_hub_download("ash56/ssl-aasist", fn), ckpt_dir / fn)

if not (REPO / "aasist_backend.py").exists():
    subprocess.run([sys.executable, "vendor_aasist.py"], cwd=REPO, check=True)

In [ ]:
sys.path.insert(0, str(REPO))
import public_ckpt_tta as pct

cfg = pct.CHECKPOINTS["ssl_aasist_wavefake"]
CROP, SR, FAKE_COL = cfg["crop"], pct.SR, cfg["fake_col"]

model, n_front, n_back = pct.build_model("ssl_aasist_wavefake", DEVICE)
print(n_front, n_back, CROP, SR)

## Decode songs into 4 s windows

In [ ]:
import soundfile as sf

EXTS = {".mp3", ".m4a", ".wav", ".flac", ".ogg", ".opus", ".aac", ".webm"}


def decode(path):
    if FFMPEG:
        cmd = [FFMPEG, "-v", "error", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", str(SR), "-"]
        out = subprocess.run(cmd, capture_output=True, check=True).stdout
        return np.frombuffer(out, dtype=np.float32).copy()
    x, sr = sf.read(str(path), dtype="float32")
    if x.ndim > 1:
        x = x.mean(1)
    if sr != SR:
        import torchaudio
        x = torchaudio.functional.resample(torch.from_numpy(x), sr, SR).numpy()
    return x


def split(x):
    if len(x) < CROP:
        return [np.tile(x, CROP // max(len(x), 1) + 1)[:CROP].astype(np.float32)]
    out = []
    for s in range(0, len(x) - CROP + 1, CROP):
        w = x[s:s + CROP]
        if np.sqrt((w ** 2).mean()) >= MIN_RMS:
            out.append(w.astype(np.float32))
        if len(out) == MAX_CHUNKS:
            break
    return out

In [ ]:
npy, idx_csv = OUT / "chunks.npy", OUT / "chunks.csv"

if npy.exists():
    chunks = np.load(npy)
    df = pd.read_csv(idx_csv)
else:
    files = sorted(p for p in SONGS.rglob("*") if p.suffix.lower() in EXTS)
    print(len(files), "files")
    wins, rows = [], []
    for i, p in enumerate(files):
        try:
            ws = split(decode(p))
        except Exception as e:
            print("skip", p.name, e)
            continue
        for j, w in enumerate(ws):
            rows.append({"file": str(p.relative_to(SONGS)), "t": j * CROP / SR})
        wins += ws
        if (i + 1) % 10 == 0:
            print(i + 1, len(wins))
    chunks, df = np.stack(wins), pd.DataFrame(rows)
    np.save(npy, chunks)
    df.to_csv(idx_csv, index=False)

buf = torch.from_numpy(chunks).half()
idx = torch.arange(len(df))
print(len(df), "windows", df.file.nunique(), "songs")

## Score

In [ ]:
t0 = time.time()
df["src"] = pct.score(model, buf, idx, FAKE_COL, BATCH, DEVICE)
print(f"{(time.time() - t0) / 60:.1f} min")


def per_song(col):
    g = df.groupby("file")[col]
    out = pd.DataFrame({
        "n": g.size(),
        "mean": g.mean(),
        "median": g.median(),
        "p90": g.quantile(0.9),
        "max": g.max(),
    })
    return out.sort_values("mean", ascending=False)


songs = per_song("src")
songs.to_csv(OUT / "songs_scores.csv")
df.to_csv(OUT / "window_scores.csv", index=False)
songs.head(20).round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(songs["mean"], bins=20)
ax[0].set_xlabel("song mean P(fake)")
ax[1].hist(df["src"], bins=50)
ax[1].set_xlabel("window P(fake)")
plt.tight_layout()
plt.savefig(OUT / "fig_scores.png", dpi=140)
plt.show()

for f in list(songs.index[:1]) + list(songs.index[-1:]):
    d = df[df.file == f]
    plt.figure(figsize=(10, 2.2))
    plt.plot(d.t, d.src, marker="o", ms=3)
    plt.ylim(0, 1)
    plt.title(f)
    plt.show()

## TTA

Pseudo-labels the top/bottom 30%, so the pool needs both kinds of song in it.

In [ ]:
s = df["src"].values
print("frac >= 0.5", (s >= 0.5).mean().round(3), "| q30-q70", np.quantile(s, [0.3, 0.7]).round(3))

if RUN_TTA:
    t0 = time.time()
    model, info = pct.adapt(model, buf, idx, FAKE_COL, BATCH, DEVICE)
    df["tta"] = pct.score(model, buf, idx, FAKE_COL, BATCH, DEVICE)
    songs_tta = per_song("tta")
    songs_tta.to_csv(OUT / "songs_scores_tta.csv")
    df.to_csv(OUT / "window_scores.csv", index=False)
    print(f"{(time.time() - t0) / 60:.1f} min, spearman",
          df[["src", "tta"]].corr(method="spearman").iloc[0, 1].round(3))
    display(songs_tta.head(20).round(3))

## EER / AUC, if labels exist

In [ ]:
from sklearn.metrics import roc_auc_score

from metrics import compute_eer

if LABELS.exists():
    lab = pd.read_csv(LABELS)
    lab = {Path(f).name: int(l) for f, l in zip(lab.file, lab.label)}
    y = np.array([lab.get(Path(f).name, -1) for f in songs.index])
    keep = y >= 0
    print(keep.sum(), "labelled")

    for col in ["src", "tta"]:
        if col not in df.columns:
            continue
        agg = per_song(col).loc[songs.index]
        for stat in ["mean", "p90", "max"]:
            sc = agg[stat].values[keep]
            eer, thr = compute_eer(y[keep], sc)
            print(f"{col} {stat}: EER {eer * 100:.2f}%  AUC {roc_auc_score(y[keep], sc):.3f}  thr {thr:.3f}")